In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from geopy.distance import geodesic
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer, PolynomialFeatures, RobustScaler
from sklearn.linear_model import LinearRegression, Ridge
import os
import fiona

In [3]:
meteo = pd.read_csv("../departement-25-doubs/data/meteostat/meteostat_doubs.csv")
doubs_data = gpd.read_file("../datasets_par_departement/departement-25-doubs-original.geojson")

FileNotFoundError: [Errno 2] No such file or directory: '../departement-25-doubs/data/meteostat/meteostat.csv'

In [16]:
meteo.columns

Index(['index', 'creneau', 'temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir',
       'wspd', 'pres', 'prec24h', 'snow24h', 'prec24h12', 'snow24h12',
       'temp16', 'dwpt16', 'rhum16', 'prcp16', 'wdir16', 'wspd16', 'prec24h16',
       'snow24h16', 'temp12', 'dwpt12', 'rhum12', 'prcp12', 'wdir12', 'wspd12',
       'temp15h', 'rhum15h', 'temp12h', 'rhum12h', 'temp24max', 'prec24veille',
       'sum_rain_last_7_days', 'sum_snow_last_7_days',
       'sum_consecutive_rainfall', 'dc', 'days_since_rain', 'ffmc', 'dmc',
       'isi', 'bui', 'fwi', 'daily_severity_rating', 'nesterov', 'munger',
       'kbdi', 'angstroem', 'latitude', 'longitude', 'year'],
      dtype='object')

In [17]:
meteo.head()

,index,creneau,temp,dwpt,rhum,prcp,snow,wdir,wspd,pres,...,bui,fwi,daily_severity_rating,nesterov,munger,kbdi,angstroem,latitude,longitude,year
0,36,2016-01-01,6.7,4.0,83.0,0.0,0.0,40.0,2.611111,1025.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,46.536680,6.047777,2016
1,36,2016-01-01,4.8,4.7,99.0,0.0,0.0,80.0,3.111111,1025.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,47.597255,6.879474,2016
2,36,2016-01-01,4.8,4.7,99.0,0.0,0.0,80.0,3.111111,1025.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,47.332111,6.546795,2016
3,36,2016-01-01,5.0,2.0,81.0,0.0,0.0,70.0,1.500000,1024.9,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,46.934395,6.380456,2016
4,36,2016-01-01,8.6,4.8,77.0,0.0,0.0,40.0,1.000000,1023.4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,47.332111,6.380456,2016


In [19]:
def calculer_duree(data, colonne_debut, colonne_fin):
    data[colonne_debut] = pd.to_datetime(data[colonne_debut], errors='coerce')
    data[colonne_fin] = pd.to_datetime(data[colonne_fin], errors='coerce')

    data['duree'] = data[colonne_fin] - data[colonne_debut]

    return data

def cyclical_encoding(X, max_value):
    return np.column_stack((
        np.sin(2 * np.pi * X / max_value),
        np.cos(2 * np.pi * X / max_value)
    ))

def season_encoding(month):
    if month in [12, 1, 2]:
        return 0  # Hiver
    elif month in [3, 4, 5]:
        return 1  # Printemps
    elif month in [6, 7, 8]:
        return 2  # Été
    else:
        return 3  # Automne

In [20]:
doubs_data = calculer_duree(doubs_data, 'date_debut', 'date_fin')
doubs_data['centroid'] = doubs_data.geometry.centroid
doubs_data['centroid_lon'] = doubs_data['centroid'].x
doubs_data['centroid_lat'] = doubs_data['centroid'].y
doubs_data['date_debut'] = pd.to_datetime(doubs_data['date_debut'], errors='coerce')
doubs_data['hour'] = doubs_data['date_debut'].dt.hour
doubs_data['date'] = pd.to_datetime(doubs_data['date'], errors='coerce')
doubs_data['month'] = doubs_data['date'].dt.month
doubs_data['weekday'] = doubs_data['date'].dt.weekday
doubs_data['season'] = doubs_data['month'].apply(season_encoding)
doubs_data[['hour_sin', 'hour_cos']] = cyclical_encoding(doubs_data['hour'], 24)
doubs_data[['month_sin', 'month_cos']] = cyclical_encoding(doubs_data['month'], 12)
doubs_data[['weekday_sin', 'weekday_cos']] = cyclical_encoding(doubs_data['weekday'], 7)

<ipython-input-20-d6e341e4f6d7>:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  doubs_data['centroid'] = doubs_data.geometry.centroid


In [27]:
doubs_data.columns

Index(['hex_id', 'raison_sortie', 'date_debut', 'date_fin', 'label', 'year',
       'date', 'departement', 'duree_minutes', 'geometry', 'duree', 'centroid',
       'centroid_lon', 'centroid_lat', 'hour', 'month', 'weekday', 'season',
       'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'weekday_sin',
       'weekday_cos'],
      dtype='object')

In [22]:
meteo['id_meteo'] = range(1, len(meteo) + 1)

meteo.head

<bound method NDFrame.head of         index    creneau  temp  dwpt  rhum  prcp  snow   wdir      wspd  \
0          36 2016-01-01   6.7   4.0  83.0   0.0   0.0   40.0  2.611111   
1          36 2016-01-01   4.8   4.7  99.0   0.0   0.0   80.0  3.111111   
2          36 2016-01-01   4.8   4.7  99.0   0.0   0.0   80.0  3.111111   
3          36 2016-01-01   5.0   2.0  81.0   0.0   0.0   70.0  1.500000   
4          36 2016-01-01   8.6   4.8  77.0   0.0   0.0   40.0  1.000000   
...       ...        ...   ...   ...   ...   ...   ...    ...       ...   
179911   2892 2024-06-28  23.6  15.9  62.0   1.0   0.0  254.0  3.611111   
179912   2892 2024-06-28  23.6  15.9  62.0   1.0   0.0  254.0  3.611111   
179913   2892 2024-06-28  23.6  15.9  62.0   1.0   0.0  254.0  3.611111   
179914   2892 2024-06-28  27.0  19.9  65.0   0.0   0.0  288.0  2.055556   
179915   2892 2024-06-28  24.7  19.5  73.0   0.0   0.0  272.0  3.000000   

          pres  ...       fwi  daily_severity_rating  nesterov  munge

In [23]:
doubs_data['date'] = pd.to_datetime(doubs_data['date'], errors='coerce')
meteo['creneau'] = pd.to_datetime(meteo['creneau'], errors='coerce')

doubs_data['centroid_lat'] = doubs_data['centroid_lat'].astype(float)
doubs_data['centroid_lon'] = doubs_data['centroid_lon'].astype(float)
meteo['latitude'] = meteo['latitude'].astype(float)
meteo['longitude'] = meteo['longitude'].astype(float)

missing_dates = []
meteo_doubs_pairs = []

for idx, row in doubs_data.iterrows():
    date_doubs = row['date']
    centroid_coords = (row['centroid_lat'], row['centroid_lon'])

    meteo_candidates = meteo[meteo['creneau'] == date_doubs]
    print(f"Date : {date_doubs}, Candidats : {len(meteo_candidates)}")

    if meteo_candidates.empty:
        missing_dates.append(row['hex_id'])
        continue

    meteo_candidates['distance'] = meteo_candidates.apply(
        lambda x: geodesic(centroid_coords, (x['latitude'], x['longitude'])).meters, axis=1
    )

    closest_match = meteo_candidates.loc[meteo_candidates['distance'].idxmin()]
    meteo_doubs_pairs.append({'hex_id': row['hex_id'], 'id_meteo': closest_match['id_meteo']})

meteo_doubs = pd.DataFrame(meteo_doubs_pairs)
print(f"Dates manquantes : {len(missing_dates)}")

Date : 2014-10-02 00:00:00, Candidats : 0
Date : 2014-10-03 00:00:00, Candidats : 0
Date : 2014-10-04 00:00:00, Candidats : 0
Date : 2014-10-04 00:00:00, Candidats : 0
Date : 2014-10-05 00:00:00, Candidats : 0
Date : 2014-11-02 00:00:00, Candidats : 0
Date : 2014-11-23 00:00:00, Candidats : 0
Date : 2014-11-30 00:00:00, Candidats : 0
Date : 2014-12-14 00:00:00, Candidats : 0
Date : 2014-12-22 00:00:00, Candidats : 0
Date : 2014-12-29 00:00:00, Candidats : 0
Date : 2015-01-10 00:00:00, Candidats : 0
Date : 2015-01-11 00:00:00, Candidats : 0
Date : 2015-02-10 00:00:00, Candidats : 0
Date : 2015-02-12 00:00:00, Candidats : 0
Date : 2015-02-19 00:00:00, Candidats : 0
Date : 2015-03-05 00:00:00, Candidats : 0
Date : 2015-03-06 00:00:00, Candidats : 0
Date : 2015-03-07 00:00:00, Candidats : 0
Date : 2015-03-07 00:00:00, Candidats : 0
Date : 2015-03-08 00:00:00, Candidats : 0
Date : 2015-03-08 00:00:00, Candidats : 0
Date : 2015-03-10 00:00:00, Candidats : 0
Date : 2015-03-11 00:00:00, Candid

<ipython-input-23-a739eb12972d>:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meteo_candidates['distance'] = meteo_candidates.apply(


Date : 2016-02-06 00:00:00, Candidats : 58
Date : 2016-02-08 00:00:00, Candidats : 58
Date : 2016-02-09 00:00:00, Candidats : 58
Date : 2016-02-18 00:00:00, Candidats : 58
Date : 2016-03-13 00:00:00, Candidats : 58
Date : 2016-03-15 00:00:00, Candidats : 58
Date : 2016-03-18 00:00:00, Candidats : 58
Date : 2016-03-18 00:00:00, Candidats : 58
Date : 2016-03-20 00:00:00, Candidats : 58
Date : 2016-03-23 00:00:00, Candidats : 58
Date : 2016-03-24 00:00:00, Candidats : 58
Date : 2016-03-24 00:00:00, Candidats : 58
Date : 2016-03-31 00:00:00, Candidats : 58
Date : 2016-03-31 00:00:00, Candidats : 58
Date : 2016-04-10 00:00:00, Candidats : 58
Date : 2016-04-20 00:00:00, Candidats : 58
Date : 2016-04-21 00:00:00, Candidats : 58
Date : 2016-04-22 00:00:00, Candidats : 58
Date : 2016-04-22 00:00:00, Candidats : 58
Date : 2016-05-05 00:00:00, Candidats : 58
Date : 2016-05-05 00:00:00, Candidats : 58
Date : 2016-05-08 00:00:00, Candidats : 58
Date : 2016-05-20 00:00:00, Candidats : 58
Date : 2016

In [30]:
for idx, row in meteo_doubs.iterrows():
    hex_id = row['hex_id']

    doubs_row = doubs_data[doubs_data['hex_id'] == hex_id].iloc[0]

    meteo_doubs.at[idx, 'duree_minutes'] = doubs_row['duree_minutes']
    meteo_doubs.at[idx, 'date'] = doubs_row['date']
    meteo_doubs.at[idx, 'departement'] = doubs_row['departement']
    meteo_doubs.at[idx, 'geometry'] = doubs_row['geometry']
    meteo_doubs.at[idx, 'centroid_lon'] = doubs_row['centroid_lon']
    meteo_doubs.at[idx, 'centroid_lat'] = doubs_row['centroid_lat']
    meteo_doubs.at[idx, 'hour_sin'] = doubs_row['hour_sin']
    meteo_doubs.at[idx, 'hour_cos'] = doubs_row['hour_cos']
    meteo_doubs.at[idx, 'month_sin'] = doubs_row['month_sin']
    meteo_doubs.at[idx, 'month_cos'] = doubs_row['month_cos']
    meteo_doubs.at[idx, 'weekday_sin'] = doubs_row['weekday_sin']
    meteo_doubs.at[idx, 'weekday_cos'] = doubs_row['weekday_cos']

    if (idx + 1) % 100 == 0:
        print(f"Ligne {idx + 1}/{len(meteo_doubs)} mise à jour...")

print("Mise à jour terminée.")

Ligne 100/2572 mise à jour...
Ligne 200/2572 mise à jour...
Ligne 300/2572 mise à jour...
Ligne 400/2572 mise à jour...
Ligne 500/2572 mise à jour...
Ligne 600/2572 mise à jour...
Ligne 700/2572 mise à jour...
Ligne 800/2572 mise à jour...
Ligne 900/2572 mise à jour...
Ligne 1000/2572 mise à jour...
Ligne 1100/2572 mise à jour...
Ligne 1200/2572 mise à jour...
Ligne 1300/2572 mise à jour...
Ligne 1400/2572 mise à jour...
Ligne 1500/2572 mise à jour...
Ligne 1600/2572 mise à jour...
Ligne 1700/2572 mise à jour...
Ligne 1800/2572 mise à jour...
Ligne 1900/2572 mise à jour...
Ligne 2000/2572 mise à jour...
Ligne 2100/2572 mise à jour...
Ligne 2200/2572 mise à jour...
Ligne 2300/2572 mise à jour...
Ligne 2400/2572 mise à jour...
Ligne 2500/2572 mise à jour...
Mise à jour terminée.


In [31]:
meteo_doubs.head()

,hex_id,id_meteo,duree_minutes,date,departement,centroid_lon,centroid_lat,hour_sin,hour_cos,month_sin,month_cos,weekday_sin,weekday_cos,geometry
0,871f82a2effffff,155,69.0,2016-01-03,departement-25-doubs,6.706744,47.355752,-0.866025,-5.000000e-01,5.000000e-01,0.866025,-0.781831,0.623490,"POLYGON ((6.690034397977122 47.35984965445364,..."
1,871f82c08ffffff,2135,94.0,2016-02-06,departement-25-doubs,5.793049,47.057733,-1.000000,-1.836970e-16,8.660254e-01,0.500000,-0.974928,-0.222521,POLYGON ((5.776456116211018 47.061715496264156...
2,871f82041ffffff,2107,53.0,2016-02-06,departement-25-doubs,6.530495,47.040158,-1.000000,-1.836970e-16,8.660254e-01,0.500000,-0.974928,-0.222521,"POLYGON ((6.513864103788434 47.04425896093811,..."
3,871f80793ffffff,2228,63.0,2015-06-06,departement-25-doubs,6.824442,47.534949,0.258819,9.659258e-01,1.224647e-16,-1.000000,-0.974928,-0.222521,"POLYGON ((6.807686254175198 47.53904667235933,..."
4,871f82c73ffffff,2274,49.0,2016-02-09,departement-25-doubs,5.868218,47.085815,-0.707107,-7.071068e-01,8.660254e-01,0.500000,0.781831,0.623490,"POLYGON ((5.851614873815588 47.08980711516611,..."


In [32]:
meteo_doubs_gdf = gpd.GeoDataFrame(meteo_doubs, geometry='geometry')

In [33]:
meteo_doubs_gdf.to_file('../datasets_par_departement/meteo_doubs.geojson', driver='GeoJSON')

In [34]:
for idx, row in meteo_doubs.iterrows():
    id_meteo = row['id_meteo']

    meteo_row = meteo[meteo['id_meteo'] == id_meteo].iloc[0]

    for col in meteo.columns:
        if col not in ['longitude', 'latitude', 'id_meteo']:
            meteo_doubs.at[idx, col] = meteo_row[col]

    if (idx + 1) % 100 == 0:
        print(f"Ligne {idx + 1}/{len(meteo_doubs)} mise à jour...")

print("Mise à jour terminée.")

Ligne 100/2572 mise à jour...
Ligne 200/2572 mise à jour...
Ligne 300/2572 mise à jour...
Ligne 400/2572 mise à jour...
Ligne 500/2572 mise à jour...
Ligne 600/2572 mise à jour...
Ligne 700/2572 mise à jour...
Ligne 800/2572 mise à jour...
Ligne 900/2572 mise à jour...
Ligne 1000/2572 mise à jour...
Ligne 1100/2572 mise à jour...
Ligne 1200/2572 mise à jour...
Ligne 1300/2572 mise à jour...
Ligne 1400/2572 mise à jour...
Ligne 1500/2572 mise à jour...
Ligne 1600/2572 mise à jour...
Ligne 1700/2572 mise à jour...
Ligne 1800/2572 mise à jour...
Ligne 1900/2572 mise à jour...
Ligne 2000/2572 mise à jour...
Ligne 2100/2572 mise à jour...
Ligne 2200/2572 mise à jour...
Ligne 2300/2572 mise à jour...
Ligne 2400/2572 mise à jour...
Ligne 2500/2572 mise à jour...
Mise à jour terminée.


In [35]:
meteo_doubs.head()

,hex_id,id_meteo,duree_minutes,date,departement,centroid_lon,centroid_lat,hour_sin,hour_cos,month_sin,...,dmc,isi,bui,fwi,daily_severity_rating,nesterov,munger,kbdi,angstroem,year
0,871f82a2effffff,155,69.0,2016-01-03,departement-25-doubs,6.706744,47.355752,-0.866025,-5.000000e-01,5.000000e-01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0
1,871f82c08ffffff,2135,94.0,2016-02-06,departement-25-doubs,5.793049,47.057733,-1.000000,-1.836970e-16,8.660254e-01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0
2,871f82041ffffff,2107,53.0,2016-02-06,departement-25-doubs,6.530495,47.040158,-1.000000,-1.836970e-16,8.660254e-01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0
3,871f80793ffffff,2228,63.0,2015-06-06,departement-25-doubs,6.824442,47.534949,0.258819,9.659258e-01,1.224647e-16,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0
4,871f82c73ffffff,2274,49.0,2016-02-09,departement-25-doubs,5.868218,47.085815,-0.707107,-7.071068e-01,8.660254e-01,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016.0


In [37]:
meteo_doubs_gdf = gpd.GeoDataFrame(meteo_doubs, geometry='geometry')
meteo_doubs_gdf.to_file('../datasets_par_departement/meteo_doubs.geojson', driver='GeoJSON')